<a href="https://colab.research.google.com/github/pop123-ux/doc-assistant-hf/blob/main/coded-test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pypdf python-docx gradio regex nltk
!pip install -q -U transformers peft
!pip install -q torch
!pip install -q -U "bitsandbytes>=0.46.1"
!pip install numpy networkx scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 12.7 MB/s eta 0:00:00


This notebook represents the pipeline of building an app where users upload PDFs, Word files, or text, and receive concise summaries or answers to specific questions about the document

- facebook/bart-large-cnn for summarization and allenai/longformer-large-4096-finetuned-triviaqa for question answering

- Toggle to switch between **abstractive** (rewriting) and **extractive** (bullet points) summarization -- future expansion

In [ ]:
!hf auth login

Hint: A new version of huggingface_hub (1.27.0) is available! You are using version 1.23.0.
To update, run: hf update
? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: 7B6H-PTIZ

    Waiting for authorization....
Token is valid.
The token `oauth-pop123ux` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-pop123ux`
Note: This token will be refreshed automatically when it expires.


In [ ]:
import json
import os
import torch
from datasets import Dataset, load_dataset
from huggingface_hub import hf_hub_download
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

dataset1 = load_dataset('knkarthick/samsum')


README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

In [ ]:
dataset1

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [ ]:
# Firstly, let's import the bart-large-cnn as for the summarization model
model1 = "facebook/bart-large-cnn"

tokenizer1 = AutoTokenizer.from_pretrained(model1, use_fast=True)
tokenizer1.pad_token = tokenizer1.eos_token
tokenizer1.padding_side = "right"

In [ ]:
# Now the tokenize function

def tokenize(examples):
  input_text = [f"{d}" for d in examples['dialogue']]

  model_inputs = tokenizer1(input_text, max_length=512, truncation=True, padding=False) # The data collator handles padding dynamically

  labels = tokenizer1(text_target=examples['summary'], max_length=128, truncation=True, padding=False)

  labels['input_ids'] = [
      [(token if token != tokenizer1.pad_token_id else -100) for token in label] for label in labels['input_ids']
  ]
  model_inputs['labels'] = labels['input_ids']

  return model_inputs

In [ ]:
tokenized_dataset = dataset1.map(tokenize, batched=True, remove_columns=dataset1['train'].column_names)
tokenized_dataset

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

In [ ]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

# Configure the 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load the model with quantization
model1 = AutoModelForSeq2SeqLM.from_pretrained(
    model1,
    quantization_config=bnb_config,
    device_map="auto",
)
model1

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear4bit(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear4bit(in_features=4096, out_features=1

In [ ]:
# Configure LoRA (PEFT)
from peft import prepare_model_for_kbit_training

model1.gradient_checkpointing_enable()

model1 = prepare_model_for_kbit_training(model1)

peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=8,
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model1 = get_peft_model(model1, peft_config)
model1.config.use_cache = False # We stop the cache during the training !
model1.print_trainable_parameters()

trainable params: 1,179,648 || all params: 407,470,080 || trainable%: 0.2895


In [ ]:
# Now the training of the first model

model1.config.use_cache = False

data_collator = DataCollatorForSeq2Seq(tokenizer1, model=model1, label_pad_token_id=-100) # label_pad_token forces padding ignore in loss calculation

training_args = Seq2SeqTrainingArguments(
    output_dir = "./lora_bart_samsum_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy='epoch',
    learning_rate=5e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    fp16=True,
    logging_steps=1,
    logging_first_step=True,
    save_strategy='epoch',
    report_to='none', # We don't want wandb logging in this case
)

trainer = Seq2SeqTrainer(
    model=model1,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset['validation'],
    processing_class=tokenizer1,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,4.141346,1.573228


In [ ]:
# Let's see the loss log history
trainer.state.log_history

[{'loss': 8.091075897216797,
  'grad_norm': nan,
  'learning_rate': 5e-05,
  'epoch': 0.0010860711376595167,
  'step': 1},
 {'loss': 9.441326141357422,
  'grad_norm': 6.256505012512207,
  'learning_rate': 4.9945711183496205e-05,
  'epoch': 0.0021721422753190334,
  'step': 2},
 {'loss': 9.4444580078125,
  'grad_norm': 7.459821701049805,
  'learning_rate': 4.98914223669924e-05,
  'epoch': 0.00325821341297855,
  'step': 3},
 {'loss': 8.028532028198242,
  'grad_norm': 5.746951580047607,
  'learning_rate': 4.9837133550488604e-05,
  'epoch': 0.004344284550638067,
  'step': 4},
 {'loss': 9.714046478271484,
  'grad_norm': 6.187074661254883,
  'learning_rate': 4.97828447339848e-05,
  'epoch': 0.005430355688297583,
  'step': 5},
 {'loss': 8.847004890441895,
  'grad_norm': 7.0356245040893555,
  'learning_rate': 4.9728555917481e-05,
  'epoch': 0.0065164268259571,
  'step': 6},
 {'loss': 8.54694652557373,
  'grad_norm': nan,
  'learning_rate': 4.96742671009772e-05,
  'epoch': 0.007602497963616617,


In [ ]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/pop123ux/lora_bart_samsum_results/commit/347b650c168b5241ad7886f74880a83b95fe4719', commit_message='End of training', commit_description='', oid='347b650c168b5241ad7886f74880a83b95fe4719', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pop123ux/lora_bart_samsum_results', endpoint='https://huggingface.co', repo_type='model', repo_id='pop123ux/lora_bart_samsum_results'), pr_revision=None, pr_num=None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

trainer.save_model("/content/drive/MyDrive/best_lora_adapter")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig, AutoModelForQuestionAnswering
from peft import PeftModel
import torch

drive.mount('/content/drive')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

adapter_path = "/content/drive/MyDrive/best_lora_adapter"
model1 = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/bart-large-cnn",
    quantization_config=bnb_config,
    device_map="auto",
)
model1

Mounted at /content/drive


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear4bit(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear4bit(in_features=4096, out_features=1

In [ ]:
tokenizer1 = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model1 = PeftModel.from_pretrained(model1, adapter_path)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.encoder.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.encoder.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.encoder.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.encoder.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.encoder.layers.1.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.encoder.layers.1.self_attn.v_proj.lora_B.default.weight', 'base_model.model.b

Now as we're done with the first model we're going to continue with the second one, which is a longformer-large model with a context window of 4096 tokens finetuned on the triviaqa dataset (contains over 650K question-answer-evidence triples)

In [ ]:
model2_name = "allenai/longformer-large-4096-finetuned-triviaqa"

model2 = AutoModelForQuestionAnswering.from_pretrained(model2_name, device_map='auto')

tokenizer2 = AutoTokenizer.from_pretrained(model2_name)

config.json:   0%|          | 0.00/866 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.74GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.74GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/535 [00:00<?, ?it/s]

[transformers] LongformerForQuestionAnswering LOAD REPORT from: allenai/longformer-large-4096-finetuned-triviaqa
Key                            | Status     |  | 
-------------------------------+------------+--+-
longformer.pooler.dense.weight | UNEXPECTED |  | 
longformer.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
tokenizer1.model_max_length

1000000000000000019884624838656

Now we're going to implement the document parsing functions and the gradio interface

In [ ]:
import gradio as gr
import os
import torch
from pypdf import PdfReader
from docx import Document
import regex as re
import nltk
import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True) # Necessary in newer versions of nltk (natural language toolkit)
except Exception as e:
    print(f"{e}")

device = 'cuda' if torch.cuda.is_available() else 'cpu' # for safety, even though the code is running without this line

def extract_text_from_file(file):
    if file is None:
        return ""

    file_path = file.name
    ext = os.path.splitext(file_path)[1].lower()
    extracted_text = ""

    try:
        if ext == ".pdf":
            reader = PdfReader(file_path)
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    extracted_text += text + "\n"

        elif ext in [".docx", ".doc"]:
            doc = Document(file_path)
            for para in doc.paragraphs:
                if para.text:
                    extracted_text += para.text + "\n"

        elif ext == ".txt":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                extracted_text = f.read()

        else:
            return "Unsupported file format. Please upload PDF, DOCX, or TXT."

        return extracted_text.strip()

    except Exception as e:
        return f"Error reading file: {e}"

def preprocess_text(text):

  original = [s.strip() for s in nltk.sent_tokenize(text) if s.strip()]
  cleaned = [re.sub(r'[^a-z0-9\s]', '', s.lower()) for s in original]
  keep  = [(o, c) for o, c in zip(original, cleaned) if c.strip()]

  return [o for o, c in keep], [c for o, c in keep]

def textrank_summarization(text, file, num_sentences):
  text = extract_text_from_file(file) if file else text

  if file:
    if not text.strip():
      return "Please enter text or upload a file."

  # Preprocess text
  original_sentences, cleaned_sentences = preprocess_text(text)

  if len(original_sentences) <= num_sentences:
    return "\n".join(f"- {s}" for s in original_sentences)

  # Compute tfidf
  tfidf = TfidfVectorizer(stop_words='english').fit_transform(cleaned_sentences)

  # Compute cosine similarity matrix
  similarity_matrix = cosine_similarity(tfidf, tfidf)

  # Avoid self-loops by setting diagonal to 0
  np.fill_diagonal(similarity_matrix, 0)

  # Create graph using NetworkX
  nx_graph = nx.from_numpy_array(similarity_matrix)

  # Run PageRank algorithm
  scores = nx.pagerank(nx_graph, alpha=0.85, max_iter=100)

  # Get top N sentences
  ranked_sentences = sorted(scores.items(), key=lambda x: x[1], reverse=True)
  top_sentence_indices = sorted(idx for idx, _ in ranked_sentences[:num_sentences])

  return "\n".join(f"- {original_sentences[i]}" for i in top_indices)


def process_summary_abstractive(text, file):
    source_text = extract_text_from_file(file) if file else text

    if not source_text.strip():
        return "Please enter text or upload a file."

    max_input_length = min(tokenizer1.model_max_length, 1024) # Bart-large CNN only has 1024 positional embeddings

    inputs = tokenizer1(
        source_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
    ).to(model1.device)

    with torch.no_grad():
        summary_ids = model1.generate(
            **inputs,
            max_new_tokens=256,
            min_new_tokens=50,
            no_repeat_ngram_size=3,
            repetition_penalty=1.15,
            length_penalty=1.2,
            num_beams=4,
            do_sample=False,
            early_stopping=True,
        )

    summary = tokenizer1.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

def process_summary_extractive(text, file, bpn):
    source_text = extract_text_from_file(file) if file else text

    assert bpn > 2, "choose a bullet point number higher than 2 for inference"
    assert bpn <= 6, "choose a bullet point number lower than 6 for inference"
    assert bpn is not None, "choose a bullet point number when using the extractive option"

    if not source_text.strip():
        return "Please enter text or upload a file."

    max_input_length = min(tokenizer1.model_max_length, 1024) # Bart-large CNN only has 1024 positional embeddings

    inputs = tokenizer1(
        source_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
    ).to(model1.device)

    with torch.no_grad():
        summary_ids = model1.generate(
            **inputs,
            max_new_tokens=35 * bpn,
            min_new_tokens=14 * bpn,
            do_sample=False,
            no_repeat_ngram_size=3,
            repetition_penalty=1.12,
            length_penalty=1.0,
            num_beams=4,
            early_stopping=True,
      )

    summary = tokenizer1.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    # implementation with nltk
    sentences = nltk.sent_tokenize(summary)

    bullet_points = []
    for i in range(int(bpn)): # int(bpn) for safety regarding the number of bullet points
      if i < len(sentences):
        clean_sentence = sentences[i].strip().lstrip("-*• ")
        bullet_points.append(f"- {clean_sentence}")
      else:
        bullet_points.append("- Key detail omitted by BART (expand input or adjust parameters).")

    summary = "\n".join(bullet_points)

    return summary


def process_qa(text, file, question):
    source_text = extract_text_from_file(file) if file else text

    if not source_text.strip():
        return "Please provide a document or text."

    if not question.strip():
        return "Please enter a question."

    inputs = tokenizer2(
        question,
        source_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(model2.device)

    with torch.no_grad():
        outputs = model2(**inputs)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]

    max_answer_length = 30
    n_best_size = 20

    start_indexes = torch.topk(start_logits, n_best_size).indices.tolist()
    end_indexes = torch.topk(end_logits, n_best_size).indices.tolist()

    best_score = -float("inf")
    best_answer = ""

    input_ids = inputs["input_ids"][0]

    for start in start_indexes:
        for end in end_indexes:

            if end < start:
                continue

            if end - start + 1 > max_answer_length:
                continue

            score = start_logits[start] + end_logits[end]

            if score > best_score:
                answer = tokenizer2.decode(
                    input_ids[start:end + 1],
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True,
                ).strip()

                if answer:
                    best_score = score
                    best_answer = answer

    if not best_answer:
        return "No answer found."

    return best_answer

In [ ]:
# Now the Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
  gr.Markdown("# 📄 Multi-Format Document Assistant")
  gr.Markdown("AI-powered portofolio tool to summarize text and extract precise answers from documents.")

  with gr.Tab("Document Summarizer (Extractive) - LIMIT: 2048 tokens"): # With the Limit set at 2048 tokens (1.3 tokens ~ 1 word) this pipeline can't process large chunks of text, but I simply couldn't finetune a larger model on a free Tesla T4 GPU
    with gr.Row():
      with gr.Column():
        sum_text = gr.Textbox(label="Option A: Paste text directly (lengthy if more bullet points selected)", lines=6, placeholder="Enter text...")
        btn_nr = gr.Number(label=" ❗(MANDATORY) Insert the number of bullet points you want (max 6)", value=6, precision=0)
        sum_file = gr.File(label="Option B: Upload Document (PDF, DOCX, TXT)", file_types=[".pdf", ".docx", ".txt"])
        summary_btn = gr.Button("Generate Summary", variant="primary")
      with gr.Column():
        summary_output = gr.Textbox(label='Summary Output', lines=6, interactive=False)

    summary_btn.click(fn=textrank_summarization, inputs=[sum_text, sum_file, btn_nr], outputs=summary_output)
  with gr.Tab("Document Summarizer (Abstractive) - LIMIT: 2048 tokens"): # - || -
    with gr.Row():
      with gr.Column():
        sum_text = gr.Textbox(label="Option A: Paste text directly", lines=6, placeholder="Enter text...")
        sum_file = gr.File(label="Option B: Upload Document (PDF, DOCX, TXT)", file_types=[".pdf", ".docx", ".txt"])
        summary_btn = gr.Button("Generate Summary", variant="primary")
      with gr.Column():
        summary_output = gr.Textbox(label='Summary Output', lines=6, interactive=False)

    summary_btn.click(fn=process_summary_abstractive, inputs=[sum_text, sum_file], outputs=summary_output)

  with gr.Tab("Document QA System"):
    with gr.Row():
      with gr.Column():
        qa_text = gr.Textbox(label="Option A: Paste context directly", lines=6, placeholder="Enter context...")
        qa_file = gr.File(label="Option B: Upload Document (PDF, DOCX, TXT)", file_types=[".pdf", ".docx", ".txt"])
        qa_question = gr.Textbox(label="Your Question", lines=2, placeholder="What is the main revenue driver mentioned?")
        qa_btn = gr.Button("Find Answer", variant="primary")
      with gr.Column():
        qa_output = gr.Textbox(label='Extracted Answer', lines=4, interactive=False)

    qa_btn.click(fn=process_qa, inputs=[qa_text, qa_file, qa_question], outputs=qa_output)

# Launch the app with a public share link
demo.launch(share=True)


/tmp/ipykernel_1019/1166783523.py:2: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://174f87c6ecaa92af23.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
